# Car detector: cloud training
Runtime > Change runtime type > GPU (T4 is fine; A100/L4 faster). Upload `car-data.zip` (from the `cloud/` folder) with the Files panel, then run all cells.

In [ ]:
!pip -q install ultralytics
!unzip -qo car-data.zip -d /content

In [ ]:
import time, shutil
from ultralytics import YOLO
BASE = '/content/yolocar-11n.pt'
COMMON = dict(data='/content/yolo/data.yaml', imgsz=1536, device=0, project='/content/runs', exist_ok=True,
              optimizer='AdamW', flipud=0.5, fliplr=0.5, degrees=0, patience=0, workers=4, plots=False, verbose=False)
# name: (weights to start from, overrides)
EXPERIMENTS = {
  'n_base':  (BASE, dict(epochs=60, lr0=0.0005, batch=8)),
  'n_long':  (BASE, dict(epochs=120, lr0=0.0005, batch=8)),
  's_coco':  ('yolo11s.pt', dict(epochs=80, lr0=0.002, batch=8)),
  'm_coco':  ('yolo11m.pt', dict(epochs=80, lr0=0.002, batch=8)),
}
for name, (start, over) in EXPERIMENTS.items():
    t = time.time()
    YOLO(start).train(name=name, **{**COMMON, **over})
    print(name, 'done in', round((time.time()-t)/60, 1), 'min')

In [ ]:
import pandas as pd
for name in EXPERIMENTS:
    df = pd.read_csv(f'/content/runs/{name}/results.csv'); df.columns = df.columns.str.strip()
    print(name, df[['metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)']].iloc[-1].round(3).to_dict())

In [ ]:
!cd /content/runs && zip -qr results.zip */weights/best.pt */results.csv
from google.colab import files; files.download('/content/runs/results.zip')